<a href="https://colab.research.google.com/github/Tshoki-m/Capstone-Research/blob/main/Copy_of_CIFAR100_TensorBoard_Metrics_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extract CIFAR-100 PEPITA run metrics

This notebook reads the TensorBoard event files in the completed `cifar100_pepita_1hidden.zip` run archive. It does not retrain the model or modify the archived experiment. It reports available scalar tags and extracts the final logged accuracy and loss from the main training event file.

The run monitored the test loader during training, so the values are final logged test metrics, not metrics from an untouched holdout.


## 1. Install the event-file reader
Run this cell once in a fresh Colab runtime.

In [ ]:
%pip -q install tensorboard


## 2. Upload the run archive
Upload `cifar100_pepita_1hidden.zip` downloaded from the experiment. If it is in Google Drive instead, mount Drive and set `ARCHIVE` to its path.

In [ ]:
from google.colab import files
uploaded = files.upload()
ARCHIVE = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if ARCHIVE is None:
    raise FileNotFoundError('Upload the cifar100_pepita_1hidden.zip archive.')
print('Using archive:', ARCHIVE)


## 3. Extract and verify the archived runtime record

In [ ]:
from pathlib import Path
import json, tempfile, zipfile

extract_root = Path(tempfile.mkdtemp(prefix='cifar100_pepita_'))
with zipfile.ZipFile(ARCHIVE) as archive:
    archive.extractall(extract_root)

runtime_files = list(extract_root.rglob('runtime_info.json'))
if runtime_files:
    runtime_info = json.loads(runtime_files[0].read_text(encoding='utf-8'))
    print(json.dumps(runtime_info, indent=2))
else:
    runtime_info = {}
    print('No runtime_info.json found in the archive.')


## 4. List scalar tags and their last logged values
The main training event file should contain the epoch-level test/validation metrics. The archive may also contain separate diagnostic events (angles, feedback statistics, and weight norms), which are not the test accuracy/loss.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

all_events = list(extract_root.rglob('events.out.tfevents.*'))
main_events = [p for p in all_events if 'version_0' in p.parts and not any(x in p.parts for x in ('angle_al_total', 'b_means_layer0', 'b_stds_layer0', 'weight_norms_layer0', 'weight_norms_layer1'))]
if not main_events:
    raise FileNotFoundError('No main TensorBoard event file found in the extracted archive.')

# There should normally be one main event file; pick the one with the most scalar tags.
loaded = []
for event_path in main_events:
    acc = EventAccumulator(str(event_path), size_guidance={'scalars': 0})
    acc.Reload()
    loaded.append((event_path, acc))
event_path, acc = max(loaded, key=lambda item: len(item[1].Tags().get('scalars', [])))
scalar_tags = acc.Tags().get('scalars', [])
print('Main event file:', event_path)
print('Scalar tags:', scalar_tags)
for tag in scalar_tags:
    events = acc.Scalars(tag)
    if events:
        last = max(events, key=lambda event: event.step)
        print(f'{tag}: step={last.step}, value={last.value:.8g}')


## 5. Extract final logged accuracy and loss
This cell identifies likely test/validation accuracy and loss tags. Review the printed scalar-tag list above if the tag names differ; edit `ACCURACY_TAG` and `LOSS_TAG` accordingly.

In [ ]:
tags = acc.Tags().get('scalars', [])

def choose_tag(kind):
    candidates = []
    for tag in tags:
        t = tag.lower()
        is_accuracy = ('acc' in t or 'accuracy' in t)
        is_loss = 'loss' in t
        if (kind == 'accuracy' and is_accuracy) or (kind == 'loss' and is_loss):
            # Prefer test/validation metrics and avoid training metrics if possible.
            score = 0
            if any(word in t for word in ('test', 'val', 'valid')): score += 4
            if any(word in t for word in ('train', 'training')): score -= 3
            candidates.append((score, tag))
    return sorted(candidates, reverse=True)[0][1] if candidates else None

ACCURACY_TAG = choose_tag('accuracy')
LOSS_TAG = choose_tag('loss')
print('Selected accuracy tag:', ACCURACY_TAG)
print('Selected loss tag:', LOSS_TAG)
if ACCURACY_TAG is None or LOSS_TAG is None:
    raise ValueError('Could not automatically identify both metrics. Use the tag list above to set ACCURACY_TAG and LOSS_TAG manually.')

def final_scalar(tag):
    events = acc.Scalars(tag)
    event = max(events, key=lambda item: item.step)
    return {'tag': tag, 'step': int(event.step), 'value': float(event.value)}

result = {
    'dataset': 'CIFAR-100',
    'method': 'PEPITA, one hidden layer, weight decay',
    'final_logged_accuracy': final_scalar(ACCURACY_TAG),
    'final_logged_loss': final_scalar(LOSS_TAG),
    'runtime': runtime_info.get('runtime', {}),
    'seed': runtime_info.get('seed'),
    'started_utc': runtime_info.get('started_utc'),
    'finished_utc': runtime_info.get('finished_utc'),
    'note': 'Test-loader metrics were monitored during training; not an untouched holdout.'
}
print(json.dumps(result, indent=2))


## 6. Save the extracted result
The JSON file can be downloaded and the accuracy/loss copied into the report. Keep the original archive as the primary run evidence.

In [ ]:
from google.colab import files

result_path = Path('/content/cifar100_final_metrics.json')
result_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
files.download(str(result_path))
